In [ ]:
#!/usr/bin/env python3
from http.server import HTTPServer, BaseHTTPRequestHandler
import json
import subprocess
import tempfile
import os
import base64
from pathlib import Path
import urllib.parse

class MLHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/':
            self.serve_html()
        else:
            self.send_error(404, "Not Found")
    
    def do_POST(self):
        if self.path == '/predict':
            self.handle_predict()
        else:
            self.send_error(404, "Not Found")
    
    def serve_html(self):
        with open("index.html", 'r') as f:
            html = f.read()
        
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.send_header('Content-Type', 'text/html')
        self.end_headers()
        self.wfile.write(html.encode('utf-8'))
    
    def handle_predict(self):
        try:
            content_length = int(self.headers.get('Content-Length', 0))
            body = self.rfile.read(content_length).decode('utf-8')

            data = json.loads(body)
            image_data = data.get('image')
            
            if not image_data:
                self.send_json_error(400, "Нет изображения")
                return
            
            try:
                # Remove data URL prefix if present
                if ',' in image_data:
                    image_data = image_data.split(',')[1]
                
                image_bytes = base64.b64decode(image_data)
            except Exception as e:
                self.send_json_error(400, f"Нет изображения: {e}")
                return
            
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
                tmp.write(image_bytes)
                image_path = tmp.name
            
            try:
                print(f"Запускаем модель на: {image_path}")

                result = subprocess.run(
                    ['python3', str("inference_seq.py"), str(image_path)],
                    capture_output=True,
                    text=True,
                    timeout=120,
                )
                
                print(f"Вывод модели: {result.stdout[:200]}")
                if result.stderr:
                    print(f"Ошибка модели: {result.stderr}")
                
                if result.returncode != 0:
                    self.send_json_error(500, f"Ошибка модели: {result.stderr}")
                    return

                self.send_json_response(200, { "image": result.stdout })
                
            finally:
                try:
                    os.unlink(image_path)
                except:
                    pass
                    
        except subprocess.TimeoutExpired:
            self.send_json_error(500, "Таймаут")
        except Exception as e:
            print(f"Ошибка: {e}")
            self.send_json_error(500, f"Ошибка сервера: {str(e)}")
    
    def send_json_response(self, status_code, data):
        response = json.dumps(data)
        self.send_response(status_code)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(response.encode('utf-8'))
    
    def send_json_error(self, status_code, message):
        self.send_json_response(status_code, {'error': message})
    
    def log_message(self, format, *args):
        print(f"{format % args}")        

def run_server(host='0.0.0.0', port=8080):
    server = HTTPServer((host, port), MLHandler)
    print(f"Сервер запущен на http://{host}:{port}")
    
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nОтключаем сервер...")
        server.shutdown()

if __name__ == '__main__':
    run_server()

Сервер запущен на http://0.0.0.0:8080


In [1]:
%%writefile inference_seq.py
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ------------------- Определение архитектуры U‑Net  -------------------
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.encoder = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        for feature in features:
            self.encoder.append(DoubleConv(in_channels, feature))
            in_channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        self.upconvs = nn.ModuleList()
        self.decoder = nn.ModuleList()
        for feature in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.decoder.append(DoubleConv(feature*2, feature))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for layer in self.encoder:
            x = layer(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        skip_connections = skip_connections[::-1]
        for idx in range(len(self.upconvs)):
            x = self.upconvs[idx](x)
            skip = skip_connections[idx]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
            x = torch.cat((skip, x), dim=1)
            x = self.decoder[idx](x)

        return torch.sigmoid(self.final_conv(x))

# ------------------- Функция инференса -------------------
def predict_and_visualize(img_pil, torch_model, device, img_size, threshold):
    """
    Загружает модель, предсказывает маску для изображения и выводит визуализацию.
    
    Аргументы:
        image_path (str): путь к входному изображению (PNG, JPG и т.д.).
        model_path (str): путь к сохранённой модели (.pth).
        img_size (tuple): размер, к которому приводится изображение перед подачей в сеть.
        threshold (float): порог бинаризации предсказания (0–1).
        device (torch.device): устройство для инференса (по умолчанию CPU).
    """
    # 1. Загрузка модели
    model = UNet(in_channels=1, out_channels=1).to(device)
    model.load_state_dict(torch_model)
    model.eval()

    # 2. Загрузка и предобработка изображения
    original_size = img_pil.size                           # (width, height)

    # Ресайз до размера, ожидаемого сетью
    img_resized = img_pil.resize(img_size, Image.BILINEAR)
    img_array = np.array(img_resized, dtype=np.float32) / 255.0
    img_tensor = torch.from_numpy(img_array).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
    img_tensor = img_tensor.to(device)

    # 3. Предсказание
    with torch.no_grad():
        pred = model(img_tensor)
        pred_mask = (pred > threshold).float().cpu().numpy().squeeze()   # (H, W)

    # 4. Визуализация
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Оригинал
    axes[0].imshow(img_pil, cmap='gray')
    axes[0].set_title('Оригинал')
    axes[0].axis('off')

    # Предсказанная маска (в размере img_size)
    axes[1].imshow(pred_mask, cmap='gray')
    axes[1].set_title('Предсказанная маска')
    axes[1].axis('off')

    # Наложение маски на оригинал (приводим маску к оригинальному размеру)
    mask_resized = Image.fromarray((pred_mask * 255).astype(np.uint8))
    mask_resized = mask_resized.resize(original_size, Image.NEAREST)
    mask_resized = np.array(mask_resized,

Overwriting inference_seq.py


In [1]:
import os
if os.path.exists("best_unet.pth"):
    print("✅ Файл есть!", os.path.getsize("best_unet.pth"), "байт")
else:
    print("❌ Файла нет!")

✅ Файл есть! 124262149 байт


In [1]:
%%writefile server.py
#!/usr/bin/env python3
from http.server import HTTPServer, BaseHTTPRequestHandler
import json
import subprocess
import tempfile
import os
import base64
from pathlib import Path
import urllib.parse

class MLHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/':
            self.serve_html()
        else:
            self.send_error(404, "Not Found")
    
    def do_POST(self):
        if self.path == '/predict':
            self.handle_predict()
        else:
            self.send_error(404, "Not Found")
    
    def serve_html(self):
        with open("index.html", 'r') as f:
            html = f.read()
        
        self.send_response(200)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.send_header('Content-Type', 'text/html')
        self.end_headers()
        self.wfile.write(html.encode('utf-8'))
    
    def handle_predict(self):
        try:
            content_length = int(self.headers.get('Content-Length', 0))
            body = self.rfile.read(content_length).decode('utf-8')

            data = json.loads(body)
            image_data = data.get('image')
            
            if not image_data:
                self.send_json_error(400, "Нет изображения")
                return
            
            try:
                # Remove data URL prefix if present
                if ',' in image_data:
                    image_data = image_data.split(',')[1]
                
                image_bytes = base64.b64decode(image_data)
            except Exception as e:
                self.send_json_error(400, f"Нет изображения: {e}")
                return
            
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
                tmp.write(image_bytes)
                image_path = tmp.name
            
            try:
                print(f"Запускаем модель на: {image_path}")

                result = subprocess.run(
                    ['python3', str("inference_seq.py"), str(image_path)],
                    capture_output=True,
                    text=True,
                    timeout=120,
                )
                
                print(f"Вывод модели: {result.stdout[:200]}")
                if result.stderr:
                    print(f"Ошибка модели: {result.stderr}")
                
                if result.returncode != 0:
                    self.send_json_error(500, f"Ошибка модели: {result.stderr}")
                    return

                self.send_json_response(200, { "image": result.stdout })
                
            finally:
                try:
                    os.unlink(image_path)
                except:
                    pass
                    
        except subprocess.TimeoutExpired:
            self.send_json_error(500, "Таймаут")
        except Exception as e:
            print(f"Ошибка: {e}")
            self.send_json_error(500, f"Ошибка сервера: {str(e)}")
    
    def send_json_response(self, status_code, data):
        response = json.dumps(data)
        self.send_response(status_code)
        self.send_header('Access-Control-Allow-Origin', '*')
        self.send_header('Access-Control-Allow-Methods', 'GET, POST, OPTIONS')
        self.send_header('Access-Control-Allow-Headers', 'Content-Type')
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(response.encode('utf-8'))
    
    def send_json_error(self, status_code, message):
        self.send_json_response(status_code, {'error': message})
    
    def log_message(self, format, *args):
        print(f"{format % args}")        

def run_server(host='0.0.0.0', port=8080):
    server = HTTPServer((host, port), MLHandler)
    print(f"Сервер запущен на http://{host}:{port}")
    
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nОтключаем сервер...")
        server.shutdown()

if __name__ == '__main__':
    run_server()

Overwriting server.py


In [1]:
%%writefile inference_seq.py
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "f6e2eb50",
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import numpy as np\n",
    "from PIL import Image\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.nn.functional as F\n",
    "import matplotlib.pyplot as plt"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "9bcbb16f",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ------------------- Определение архитектуры U‑Net  -------------------\n",
    "class DoubleConv(nn.Module):\n",
    "    def __init__(self, in_ch, out_ch):\n",
    "        super(DoubleConv, self).__init__()\n",
    "        self.conv = nn.Sequential(\n",
    "            nn.Conv2d(in_ch, out_ch, 3, padding=1),\n",
    "            nn.BatchNorm2d(out_ch),\n",
    "            nn.ReLU(inplace=True),\n",
    "            nn.Conv2d(out_ch, out_ch, 3, padding=1),\n",
    "            nn.BatchNorm2d(out_ch),\n",
    "            nn.ReLU(inplace=True)\n",
    "        )\n",
    "\n",
    "    def forward(self, x):\n",
    "        return self.conv(x)\n",
    "\n",
    "class UNet(nn.Module):\n",
    "    def __init__(self, in_channels=1, out_channels=1, features=[64, 128, 256, 512]):\n",
    "        super(UNet, self).__init__()\n",
    "        self.encoder = nn.ModuleList()\n",
    "        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)\n",
    "\n",
    "        for feature in features:\n",
    "            self.encoder.append(DoubleConv(in_channels, feature))\n",
    "            in_channels = feature\n",
    "\n",
    "        self.bottleneck = DoubleConv(features[-1], features[-1]*2)\n",
    "\n",
    "        self.upconvs = nn.ModuleList()\n",
    "        self.decoder = nn.ModuleList()\n",
    "        for feature in reversed(features):\n",
    "            self.upconvs.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))\n",
    "            self.decoder.append(DoubleConv(feature*2, feature))\n",
    "\n",
    "        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)\n",
    "\n",
    "    def forward(self, x):\n",
    "        skip_connections = []\n",
    "\n",
    "        for layer in self.encoder:\n",
    "            x = layer(x)\n",
    "            skip_connections.append(x)\n",
    "            x = self.pool(x)\n",
    "\n",
    "        x = self.bottleneck(x)\n",
    "\n",
    "        skip_connections = skip_connections[::-1]\n",
    "        for idx in range(len(self.upconvs)):\n",
    "            x = self.upconvs[idx](x)\n",
    "            skip = skip_connections[idx]\n",
    "            if x.shape != skip.shape:\n",
    "                x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)\n",
    "            x = torch.cat((skip, x), dim=1)\n",
    "            x = self.decoder[idx](x)\n",
    "\n",
    "        return torch.sigmoid(self.final_conv(x))\n",
    "\n",
    "# ------------------- Функция инференса -------------------\n",
    "def predict_and_visualize(image_path, model_path, img_size=(256, 256), threshold=0.5, device=None):\n",
    "    \"\"\"\n",
    "    Загружает модель, предсказывает маску для изображения и выводит визуализацию.\n",
    "    \n",
    "    Аргументы:\n",
    "        image_path (str): путь к входному изображению (PNG, JPG и т.д.).\n",
    "        model_path (str): путь к сохранённой модели (.pth).\n",
    "        img_size (tuple): размер, к которому приводится изображение перед подачей в сеть.\n",
    "        threshold (float): порог бинаризации предсказания (0–1).\n",
    "        device (torch.device): устройство для инференса (по умолчанию CPU).\n",
    "    \"\"\"\n",
    "    if device is None:\n",
    "        device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n",
    "\n",
    "    # 1. Загрузка модели\n",
    "    model = UNet(in_channels=1, out_channels=1).to(device)\n",
    "    model.load_state_dict(torch.load(model_path, map_location=device))\n",
    "    model.eval()\n",
    "    print(f\"Модель загружена из {model_path}\")\n",
    "\n",
    "    # 2. Загрузка и предобработка изображения\n",
    "    img_pil = Image.open(image_path).convert('L')          # ч/б\n",
    "    original_size = img_pil.size                           # (width, height)\n",
    "\n",
    "    # Ресайз до размера, ожидаемого сетью\n",
    "    img_resized = img_pil.resize(img_size, Image.BILINEAR)\n",
    "    img_array = np.array(img_resized, dtype=np.float32) / 255.0\n",
    "    img_tensor = torch.from_numpy(img_array).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)\n",
    "    img_tensor = img_tensor.to(device)\n",
    "\n",
    "    # 3. Предсказание\n",
    "    with torch.no_grad():\n",
    "        pred = model(img_tensor)\n",
    "        pred_mask = (pred > threshold).float().cpu().numpy().squeeze()   # (H, W)\n",
    "\n",
    "    # 4. Визуализация\n",
    "    fig, axes = plt.subplots(1, 3, figsize=(15, 5))\n",
    "\n",
    "    # Оригинал\n",
    "    axes[0].imshow(img_pil, cmap='gray')\n",
    "    axes[0].set_title('Оригинал')\n",
    "    axes[0].axis('off')\n",
    "\n",
    "    # Предсказанная маска (в размере img_size)\n",
    "    axes[1].imshow(pred_mask, cmap='gray')\n",
    "    axes[1].set_title('Предсказанная маска')\n",
    "    axes[1].axis('off')\n",
    "\n",
    "    # Наложение маски на оригинал (приводим маску к оригинальному размеру)\n",
    "    mask_resized = Image.fromarray((pred_mask * 255).astype(np.uint8))\n",
    "    mask_resized = mask_resized.resize(original_size, Image.NEAREST)\n",
    "    mask_resized = np.array(mask_resized, dtype=np.float32) / 255.0\n",
    "\n",
    "    # Преобразуем оригинал в RGB для цветного наложения\n",
    "    overlay = np.array(img_pil.convert('RGB'))\n",
    "    mask_bool = mask_resized > 0.5\n",
    "    # Накладываем красный полупрозрачный слой\n",
    "    overlay[mask_bool] = (overlay[mask_bool] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)\n",
    "\n",
    "    axes[2].imshow(overlay)\n",
    "    axes[2].set_title('Наложение маски')\n",
    "    axes[2].axis('off')\n",
    "\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "90e0963b",
   "metadata": {
    "scrolled": True
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Модель загружена из best_unet.pth\n"
     ]
    },
    {
     "data": {
      "image/png": "...\n",
      "text/plain": [
       "<Figure size 1080x360 with 3 Axes>"
      ]
     },
     "metadata": {
      "needs_background": "light"
     },
     "output_type": "display_data"
    }
   ],
   "source": [
    "\n",
    "    \n",
    "# ------------------- Пример использования -------------------\n",
    "if __name__ == '__main__':\n",
    "    # Укажите свои пути\n",
    "    IMAGE_PATH = \"Glioma/720.png\"           # путь к тестовому изображению\n",
    "    MODEL_PATH = \"best_unet.pth\"            # путь к обученной модели\n",
    "\n",
    "    # Запуск инференса\n",
    "    predict_and_visualize(IMAGE_PATH, MODEL_PATH, img_size=(256, 256), threshold=0.5)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "fa42d377",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}


Overwriting inference_seq.py


In [1]:
%%writefile index.html
<!DOCTYPE html>
<html>
... (весь HTML)
</html>

Overwriting index.html


In [1]:
import os
print(os.listdir())

['5', 'app_v3.tar', 'diff', 'test.py', 'etane.gro', 'topol1.top', 'pug.ipynb', '.chemaxon', '.nvidia-settings-rc', '.java', '.cache', 'plumed_2.9.0-1_amd64.deb', 'similarity.ipynb', 'M69BI-25', 'pymol.ipynb', 'docked_out', '.DEG_funkan.zip', 'PaDEL-Descriptor v2.21(1).zip', '.pymol', 'webb.ipynb', '.fltk', 'index.html', 'protein.pdb', 'knime_5.11.0', 'out.pdbqt', 'a.out', 'qsar.ipynb', 'for_dasha', 'petrushov', 'PaDEL-Descriptor v2.21(1)', 'protein_chem', 'Музыка', 'templig.pdb', 'chimera-1.17.3-linux_x86_64.bin', 'pr_g.zip', '.ssh', 'pdbqt', 'Untitled2.ipynb', 'Untitled3.ipynb', '.chimera', 'orca', 'plumed_2.9.0-1_amd64.deb.3', '.dotnet', '.python_history', 'drugs.sdf.zip', 'Изображения', '.knime', 'sofia', 'нугуманова_дз', 'PyRTL', 'tempexp.pdb', 'ext_images', 'Untitled1.ipynb', 'tundra.pdf', 'mol_2', '.config', 'proect', 'RFdiffusion', 'lig', 'gromacs-2022.6_1-1_amd64.deb', 'Untitled4.ipynb', '.vnc', '.PaDEL-Descriptor', 'PAMPA', 'Видео', 'human_gem', '4djx', 'GSE165599_NT_vs_R4_C2.

In [1]:
# Проверьте, что файлы на месте
import os
files = ['server.py', 'inference_seq.py', 'best_unet.pth', 'index.html']
for f in files:
    if os.path.exists(f):
        print(f"✅ {f}")
    else:
        print(f"❌ {f} - НЕТ!")

✅ server.py
✅ inference_seq.py
✅ best_unet.pth
✅ index.html


In [1]:
import os
if os.path.exists("index.html"):
    os.remove("index.html")
    print("✅ Старый файл удалён")

✅ Старый файл удалён


In [1]:
%%writefile index.html
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Распознавание изображений - U-Net</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 800px;
            margin: 50px auto;
            text-align: center;
            background: #f0f0f0;
        }
        .container {
            background: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }
        #inputArea {
            margin: 20px 0;
            padding: 20px;
            border: 2px dashed #ccc;
            border-radius: 10px;
            cursor: pointer;
        }
        #inputArea:hover {
            border-color: #007bff;
        }
        button {
            padding: 12px 30px;
            background: #007bff;
            color: white;
            border: none;
            border-radius: 5px;
            font-size: 16px;
            cursor: pointer;
            transition: background 0.3s;
        }
        button:hover {
            background: #0056b3;
        }
        button:disabled {
            background: #6c757d;
            cursor: not-allowed;
        }
        #result {
            margin-top: 20px;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 10px;
        }
        #result img {
            max-width: 100%;
            border-radius: 5px;
        }
        #loading {
            display: none;
            color: #007bff;
            margin: 20px 0;
        }
        #preview {
            max-width: 200px;
            margin: 10px auto;
            border-radius: 5px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🧠 Сегментация изображений</h1>
        <p>Загрузите изображение для сегментации с помощью U-Net</p>
        
        <div id="inputArea" onclick="document.getElementById('imageInput').click()">
            <input type="file" id="imageInput" accept="image/*" style="display:none">
            <p>📁 Нажмите, чтобы выбрать изображение</p>
            <img id="preview" style="display:none">
        </div>
        
        <button id="predictBtn" onclick="predict()" disabled>Распознать</button>
        
        <div id="loading">⏳ Обработка...</div>
        
        <div id="result"></div>
    </div>

    <script>
        const input = document.getElementById('imageInput');
        const preview = document.getElementById('preview');
        const predictBtn = document.getElementById('predictBtn');
        const resultDiv = document.getElementById('result');
        const loading = document.getElementById('loading');
        
        input.addEventListener('change', function(e) {
            if (this.files[0]) {
                const reader = new FileReader();
                reader.onload = function(e) {
                    preview.src = e.target.result;
                    preview.style.display = 'block';
                    predictBtn.disabled = false;
                };
                reader.readAsDataURL(this.files[0]);
            }
        });
        
        async function predict() {
            if (!input.files[0]) {
                alert('Выберите изображение');
                return;
            }
            
            const reader = new FileReader();
            reader.onload = async function(e) {
                const imageData = e.target.result;
                
                predictBtn.disabled = True;
                loading.style.display = 'block';
                resultDiv.innerHTML = '';
                
                try {
                    const response = await fetch('http://localhost:8080/predict', {
                        method: 'POST',
                        headers: {
                            'Content-Type': 'application/json',
                        },
                        body: JSON.stringify({ image: imageData })
                    });
                    
                    const result = await response.json();
                    
                    if (result.error) {
                        resultDiv.innerHTML = `<p style="color:red">❌ Ошибка: ${result.error}</p>`;
                    } else if (result.image) {
                        resultDiv.innerHTML = `
                            <h3>Результат сегментации:</h3>
                            <img src="data:image/png;base64,${result.image}" alt="Результат">
                        `;
                    }
                } catch (error) {
                    resultDiv.innerHTML = `<p style="color:red">❌ Ошибка: ${error.message}</p>`;
                } finally {
                    loading.style.display = 'none';
                    predictBtn.disabled = false;
                }
            };
            reader.readAsDataURL(input.files[0]);
        }
    </script>
</body>
</html>

Writing index.html


In [1]:
# Проверьте, что файлы на месте
import os
files = ['server.py', 'inference_seq.py', 'best_unet.pth', 'index.html']
for f in files:
    if os.path.exists(f):
        print(f"✅ {f}")
    else:
        print(f"❌ {f} - НЕТ!")

✅ server.py
✅ inference_seq.py
✅ best_unet.pth
✅ index.html
